# Свой метод прунинга: полный прогон

Перед запуском загрузите `coursework_sem7.zip` (собирает `scripts/pack_for_colab.sh`) в корень «Мой диск» и выберите среду с GPU. Результаты пишутся в `coursework_sem7/runs/` на Диске. Если сессия оборвалась, выполните ячейки заново: расчёт продолжится с места остановки.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, zipfile, pathlib
DRIVE = pathlib.Path('/content/drive/MyDrive')
PROJ = DRIVE / 'coursework_sem7'
ZIP = DRIVE / 'coursework_sem7.zip'
if not (PROJ / 'src' / 'own_method.py').exists():
    assert ZIP.exists(), f'загрузите {ZIP.name} в корень «Мой диск»'
    with zipfile.ZipFile(ZIP) as z:
        z.extractall(DRIVE)
need = ['src/own_method.py', 'data/cache/train_256.npz', 'data/cache/val_256.npz', 'data/cache/test_256.npz',
        'data/cache/ph2_256.npz', 'runs/models/base.pth', 'runs/models/unstructured_l1_90.pth']
for p in need:
    print(('ok  ' if (PROJ / p).exists() else 'НЕТ ') + p)
(PROJ / 'runs' / 'logs').mkdir(parents=True, exist_ok=True)
os.environ['PROJ'] = str(PROJ)

In [ ]:
!pip -q install segmentation-models-pytorch==0.5.0 torch-pruning==1.6.0 onnx onnxruntime
from importlib.metadata import version
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '| smp', version('segmentation-models-pytorch'), '| tp', version('torch-pruning'))

Быстрая проверка (16 картинок, вывод в `runs_smoke/`).

In [ ]:
!cd "$PROJ" && python -u src/own_method.py --smoke --levels 0.5 0.9 2>&1 | tail -n 40
!cd "$PROJ" && python -u src/own_report.py --smoke | head -n 30

Полный прогон, лог в `runs/logs/own_method_colab.log`.

In [ ]:
!cd "$PROJ" && python -u src/own_method.py 2>&1 | tee -a runs/logs/own_method_colab.log | grep --line-buffered -E "=====|---|->|Traceback|Error"

Прогресс.

In [ ]:
import csv, collections
rows = list(csv.DictReader(open(PROJ / 'runs/results/own/own_results.csv')))
done = collections.Counter((r['level'], r['method']) for r in rows if r['stage'] == 'finetuned')
for (L, m), n in sorted(done.items(), key=lambda x: (int(x[0][0]), x[0][1])):
    print(f'{L:>3}%  {m:15s} зёрен: {n}')
print('моделей:', len(list((PROJ / 'runs/models/own').glob('*.pth'))))
!tail -n 5 "$PROJ/runs/logs/own_method_colab.log"

Отчёт без замеров скорости.

In [ ]:
!cd "$PROJ" && python -u src/own_report.py | head -n 60